# OCTA Dataset Preparation


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

In [2]:
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()          
DATA_DIR     = NOTEBOOK_DIR.parent       
PROJECT_DIR  = DATA_DIR.parent           

MASTER_CSV = PROJECT_DIR.parent / "01_Data_Preprocessing" / "master_dataset.csv"
DATA_FOLDER = DATA_DIR
OUTPUT      = DATA_DIR / "master_excels" / "master_table_new.xlsx"

## 1. Načítanie a filtrovanie

In [3]:
df = pd.read_csv(MASTER_CSV)
df.columns = [c.strip() for c in df.columns]

data_folder = Path(DATA_FOLDER)
df = df[df["image_path"].apply(
    lambda p: (data_folder / str(p).strip()).exists()
    if pd.notna(p) and str(p).strip() != "" else False
)].copy().reset_index(drop=True)

print(f"Riadkov po filtrovaní: {len(df)}")

Riadkov po filtrovaní: 5741


In [4]:
# Oprava image_path: octagon_damaged\ → data/
data_folder = Path(DATA_FOLDER)

for idx, row in df.iterrows():
    path = str(row["image_path"]).strip()
    if path.startswith("octagon_damaged"):
        filename = Path(path).name
        if (data_folder / filename).exists():
            df.at[idx, "image_path"] = f"data/{filename}"

print(f"Octagon cesty opravené")

Octagon cesty opravené


## 2. Pomocné funkcie

In [5]:
def normalize_label_raw(disease):
    if pd.isna(disease) or str(disease).strip() == "" or str(disease).strip().lower() == "nan":
        return "Unknown"
    d = str(disease).strip()
    if d.upper() == "NORMAL":
        return "Healthy"
    return d

def clean_label(disease):
    if pd.isna(disease) or str(disease).strip() == "" or str(disease).strip().lower() == "nan":
        return "EXCLUDE"
    d = str(disease).strip().upper()
    if d == "DR":            return "DR"
    if d == "AMD":           return "AMD"
    if d in ["NORMAL", "HEALTHY"]: return "Healthy"
    if d in ["BRVO", "RVO", "DME,RVO"]: return "RVO"
    return "EXCLUDE"

## 3. Párovanie SVP + DCP

In [6]:
records = []

# --- OCTA-500: párovanie podľa patient_id ---
octa500     = df[df["dataset"] == "OCTA-500"].copy()
octa500_svp = octa500[octa500["modality"] == "SVP"]
octa500_dcp = octa500[octa500["modality"] == "DCP"]
octa500_dcp_dict = octa500_dcp.set_index("patient_id")["image_path"].to_dict()

for _, row in octa500_svp.iterrows():
    pid      = row["patient_id"]
    dcp_path = octa500_dcp_dict.get(pid, "")
    records.append({
        "sample_id":          f"OCTA-500_{pid}",
        "dataset":            row["dataset"],
        "patient_id":         pid,
        "svp_path":           row["image_path"],
        "dcp_path":           dcp_path,
        "has_dcp":            1 if dcp_path != "" else 0,
        "label_raw":          normalize_label_raw(row["disease"]),
        "label_clean_5class": clean_label(row["disease"]),
    })

octa500_svp_pids = set(octa500_svp["patient_id"])
for _, row in octa500_dcp.iterrows():
    if row["patient_id"] not in octa500_svp_pids:
        records.append({
            "sample_id":          f"OCTA-500_{row['patient_id']}",
            "dataset":            row["dataset"],
            "patient_id":         row["patient_id"],
            "svp_path":           "",
            "dcp_path":           row["image_path"],
            "has_dcp":            2,
            "label_raw":          normalize_label_raw(row["disease"]),
            "label_clean_5class": clean_label(row["disease"]),
        })

# --- Octagon Healthy: párovanie podľa patient_id ---
octagon_healthy = df[(df["dataset"] == "Octagon") & (df["disease"].astype(str).str.lower() == "healthy")].copy()
oct_h_svp = octagon_healthy[octagon_healthy["modality"] == "SVP"]
oct_h_dcp = octagon_healthy[octagon_healthy["modality"] == "DCP"]
oct_h_dcp_dict = oct_h_dcp.set_index("patient_id")["image_path"].to_dict()

for _, row in oct_h_svp.iterrows():
    pid      = row["patient_id"]
    dcp_path = oct_h_dcp_dict.get(pid, "")
    records.append({
        "sample_id":          f"Octagon_{pid}",
        "dataset":            row["dataset"],
        "patient_id":         pid,
        "svp_path":           row["image_path"],
        "dcp_path":           dcp_path,
        "has_dcp":            1 if dcp_path != "" else 0,
        "label_raw":          normalize_label_raw(row["disease"]),
        "label_clean_5class": clean_label(row["disease"]),
    })

oct_h_svp_pids = set(oct_h_svp["patient_id"])
for _, row in oct_h_dcp.iterrows():
    if row["patient_id"] not in oct_h_svp_pids:
        records.append({
            "sample_id":          f"Octagon_{row['patient_id']}",
            "dataset":            row["dataset"],
            "patient_id":         row["patient_id"],
            "svp_path":           "",
            "dcp_path":           row["image_path"],
            "has_dcp":            2,
            "label_raw":          normalize_label_raw(row["disease"]),
            "label_clean_5class": clean_label(row["disease"]),
        })

# --- Octagon DR: SVP samostatne (has_dcp=0), DCP samostatne (has_dcp=2) ---
octagon_dr     = df[(df["dataset"] == "Octagon") & (df["disease"].astype(str).str.upper() == "DR")].copy()
octagon_dr_svp = octagon_dr[octagon_dr["modality"] == "SVP"]
octagon_dr_dcp = octagon_dr[octagon_dr["modality"] == "DCP"]

for _, row in octagon_dr_svp.iterrows():
    records.append({
        "sample_id":          f"Octagon_{row['patient_id']}",
        "dataset":            row["dataset"],
        "patient_id":         row["patient_id"],
        "svp_path":           row["image_path"],
        "dcp_path":           "",
        "has_dcp":            0,
        "label_raw":          normalize_label_raw(row["disease"]),
        "label_clean_5class": clean_label(row["disease"]),
    })

for _, row in octagon_dr_dcp.iterrows():
    records.append({
        "sample_id":          f"Octagon_{row['patient_id']}",
        "dataset":            row["dataset"],
        "patient_id":         row["patient_id"],
        "svp_path":           "",
        "dcp_path":           row["image_path"],
        "has_dcp":            2,
        "label_raw":          normalize_label_raw(row["disease"]),
        "label_clean_5class": clean_label(row["disease"]),
    })

# --- M3OCTA: párovanie podľa patient_id + číslo merania ---
m3octa = df[df["dataset"] == "M3OCTA"].copy()

def get_m3octa_key(sample_id):
    parts = str(sample_id).split("_")
    try:
        return f"{parts[1]}_{parts[2]}_{parts[3]}"
    except:
        return sample_id

m3octa["pair_key"] = m3octa["sample_id"].apply(get_m3octa_key)
m3octa_svp = m3octa[m3octa["modality"] == "SVP"]
m3octa_dcp = m3octa[m3octa["modality"] == "DCP"]
m3octa_dcp_dict = m3octa_dcp.set_index("pair_key")["image_path"].to_dict()

for _, row in m3octa_svp.iterrows():
    key      = row["pair_key"]
    dcp_path = m3octa_dcp_dict.get(key, "")
    records.append({
        "sample_id":          f"M3OCTA_{key}",
        "dataset":            row["dataset"],
        "patient_id":         row["patient_id"],
        "svp_path":           row["image_path"],
        "dcp_path":           dcp_path,
        "has_dcp":            1 if dcp_path != "" else 0,
        "label_raw":          normalize_label_raw(row["disease"]),
        "label_clean_5class": clean_label(row["disease"]),
    })

m3octa_svp_keys = set(m3octa_svp["pair_key"])
for _, row in m3octa_dcp.iterrows():
    if row["pair_key"] not in m3octa_svp_keys:
        records.append({
            "sample_id":          f"M3OCTA_{row['pair_key']}",
            "dataset":            row["dataset"],
            "patient_id":         row["patient_id"],
            "svp_path":           "",
            "dcp_path":           row["image_path"],
            "has_dcp":            2,
            "label_raw":          normalize_label_raw(row["disease"]),
            "label_clean_5class": clean_label(row["disease"]),
        })

# --- Ostatné datasety: len SVP ---
other_datasets = df[~df["dataset"].isin(["OCTA-500", "Octagon", "M3OCTA"])].copy()
other_svp = other_datasets[other_datasets["modality"] != "DCP"]

for _, row in other_svp.iterrows():
    records.append({
        "sample_id":          row["sample_id"],
        "dataset":            row["dataset"],
        "patient_id":         row["patient_id"],
        "svp_path":           row["image_path"],
        "dcp_path":           "",
        "has_dcp":            0,
        "label_raw":          normalize_label_raw(row["disease"]),
        "label_clean_5class": clean_label(row["disease"]),
    })

result = pd.DataFrame(records)
print(f"Celkový počet riadkov: {len(result)}")
print(result["dataset"].value_counts().to_string())

Celkový počet riadkov: 4171
dataset
M3OCTA      1503
Soul        1013
Drac         960
Fazid        304
OCTA-500     300
Mosaic        52
Octagon       39


## 4. Split: Full (všetky snímky, 70/15/15 per choroba)

- `use_for_cls_full`: 1 ak má platnú triedu (DR/AMD/Healthy/RVO), `has_dcp != 2`; pre Soul RVO len pacienti začínajúci na S1
- `split_full`: train/test/val na úrovni pacientov, seed=42

In [7]:
df = result.copy()

mask_base = (
    (~df["label_clean_5class"].isin(["EXCLUDED", "EXCLUDE", ""])) &
    (df["has_dcp"] != 2)
)
soul_rvo  = (df["dataset"] == "Soul") & (df["label_clean_5class"] == "RVO") & (df["patient_id"].astype(str).str.startswith("S1"))
other_rvo = (df["label_clean_5class"] == "RVO") & (df["dataset"] != "Soul")
not_rvo   = df["label_clean_5class"] != "RVO"

mask_use = mask_base & (not_rvo | soul_rvo | other_rvo)
df["use_for_cls_full"] = mask_use.astype(int)
df["split_full"] = ""

for disease in df[mask_use]["label_clean_5class"].unique():
    sub      = df[(df["label_clean_5class"] == disease) & (df["use_for_cls_full"] == 1)]
    patients = sub["patient_id"].unique()
    train_p, temp_p = train_test_split(patients.to_numpy(), test_size=0.30, random_state=42)
    test_p,  val_p  = train_test_split(temp_p, test_size=0.50, random_state=42)
    df.loc[sub[sub["patient_id"].isin(train_p)].index, "split_full"] = "train"
    df.loc[sub[sub["patient_id"].isin(test_p)].index,  "split_full"] = "test"
    df.loc[sub[sub["patient_id"].isin(val_p)].index,   "split_full"] = "val"

print(df[mask_use].groupby(["label_clean_5class", "split_full"]).size().unstack(fill_value=0).to_string())

split_full          test  train  val
label_clean_5class                  
AMD                    6     30    7
DR                   186    862  190
Healthy               93    518   87
RVO                   82    345   70


## 5. Split: SVP+DCP (min. 50% kombinovaných, round-robin SVP only)

- `use_for_cls_svp_dcp`: 1 ak má platnú triedu; kombinované (`has_dcp=1`) + SVP only doplnené round-robinom po 1 snímke z každého datasetu, max 50% SVP only
- `split_svp_dcp`: train/test/val na úrovni pacientov, seed=42

In [8]:
VALID_DISEASES = ["DR", "AMD", "Healthy", "RVO"]

mask_base = (
    df["label_clean_5class"].isin(VALID_DISEASES) &
    (df["has_dcp"] != 2)
)
soul_rvo  = (df["dataset"] == "Soul") & (df["label_clean_5class"] == "RVO") & (df["patient_id"].astype(str).str.startswith("S1"))
other_rvo = (df["label_clean_5class"] == "RVO") & (df["dataset"] != "Soul")
not_rvo   = df["label_clean_5class"] != "RVO"
mask_use  = mask_base & (not_rvo | soul_rvo | other_rvo)

df["use_for_cls_svp_dcp"] = 0
df["split_svp_dcp"] = ""

for disease in VALID_DISEASES:
    sub        = df[mask_use & (df["label_clean_5class"] == disease)]
    combined   = sub[sub["has_dcp"] == 1]
    svp_only   = sub[sub["has_dcp"] == 0]
    n_combined = len(combined)

    svp_by_dataset = {ds: grp.index.tolist() for ds, grp in svp_only.groupby("dataset")}
    datasets  = list(svp_by_dataset.keys())
    pointers  = {ds: 0 for ds in datasets}
    selected_svp_idx = []

    stop = False
    while not stop:
        added = False
        for ds in datasets:
            if pointers[ds] >= len(svp_by_dataset[ds]): continue
            if len(selected_svp_idx) + 1 > n_combined:
                stop = True; break
            selected_svp_idx.append(svp_by_dataset[ds][pointers[ds]])
            pointers[ds] += 1
            added = True
        if not added: break

    selected_idx = combined.index.tolist() + selected_svp_idx
    df.loc[selected_idx, "use_for_cls_svp_dcp"] = 1

mask_svp_dcp = df["use_for_cls_svp_dcp"] == 1

for disease in VALID_DISEASES:
    sub      = df[(df["label_clean_5class"] == disease) & mask_svp_dcp]
    patients = sub["patient_id"].unique()
    train_p, temp_p = train_test_split(patients.to_numpy(), test_size=0.30, random_state=42)
    test_p,  val_p  = train_test_split(temp_p,              test_size=0.50, random_state=42)
    df.loc[sub[sub["patient_id"].isin(train_p)].index, "split_svp_dcp"] = "train"
    df.loc[sub[sub["patient_id"].isin(test_p)].index,  "split_svp_dcp"] = "test"
    df.loc[sub[sub["patient_id"].isin(val_p)].index,   "split_svp_dcp"] = "val"

print(df[mask_svp_dcp].groupby(["label_clean_5class", "split_svp_dcp"]).size().unstack(fill_value=0).to_string())

split_svp_dcp       test  train  val
label_clean_5class                  
AMD                    6     30    7
DR                    44    198   46
Healthy               93    518   87
RVO                   15     73   10


## 6. Split: SVP+DCP Balanced (vyvážené na počet pacientov DR)

- `use_for_cls_svp_dcp_ballanced`: triedy vyvážené na počet unikátnych pacientov DR; kombinované vyberané round-robinom po 1 snímke na pacienta; SVP only doplnené round-robinom po 1 snímke z datasetu
- `split_svp_dcp_ballanced`: train/test/val na úrovni pacientov, seed=42

In [9]:
df["use_for_cls_svp_dcp_ballanced"] = 0
df["split_svp_dcp_ballanced"] = ""

def round_robin_combined(combined, n_max):
    by_patient = {pid: grp.index.tolist() for pid, grp in combined.groupby("patient_id")}
    patients   = list(by_patient.keys())
    pointers   = {pid: 0 for pid in patients}
    selected   = []
    seen_pats  = set()
    stop = False
    while not stop:
        added = False
        for pid in patients:
            if pointers[pid] >= len(by_patient[pid]): continue
            if pid not in seen_pats and len(seen_pats) + 1 > n_max:
                stop = True; break
            selected.append(by_patient[pid][pointers[pid]])
            seen_pats.add(pid)
            pointers[pid] += 1
            added = True
        if not added: break
    return selected

def round_robin_svp(svp_only, n_combined, current_patients, n_max):
    by_dataset = {ds: grp.index.tolist() for ds, grp in svp_only.groupby("dataset")}
    datasets   = list(by_dataset.keys())
    pointers   = {ds: 0 for ds in datasets}
    selected   = []
    cur_pats   = set(current_patients)
    stop = False
    while not stop:
        added = False
        for ds in datasets:
            if pointers[ds] >= len(by_dataset[ds]): continue
            idx      = by_dataset[ds][pointers[ds]]
            pid      = df.loc[idx, "patient_id"]
            new_pats = cur_pats | {pid}
            if len(new_pats) > n_max: continue
            if len(selected) + 1 > n_combined:
                stop = True; break
            selected.append(idx)
            cur_pats = new_pats
            pointers[ds] += 1
            added = True
        if not added: break
    return selected

# Zisti n_dr
dr_sub      = df[mask_use & (df["label_clean_5class"] == "DR")]
dr_comb_idx = round_robin_combined(dr_sub[dr_sub["has_dcp"] == 1], n_max=99999)
dr_svp_idx  = round_robin_svp(dr_sub[dr_sub["has_dcp"] == 0], len(dr_comb_idx), df.loc[dr_comb_idx, "patient_id"].unique(), n_max=99999)
n_dr        = df.loc[dr_comb_idx + dr_svp_idx, "patient_id"].nunique()
print(f"DR referencia: {n_dr} unikátnych pacientov")

for disease in VALID_DISEASES:
    sub      = df[mask_use & (df["label_clean_5class"] == disease)]
    combined = sub[sub["has_dcp"] == 1]
    svp_only = sub[sub["has_dcp"] == 0]

    comb_idx      = round_robin_combined(combined, n_dr)
    comb_patients = df.loc[comb_idx, "patient_id"].unique()
    svp_idx       = round_robin_svp(svp_only, len(comb_idx), comb_patients, n_dr)

    selected_idx = comb_idx + svp_idx
    df.loc[selected_idx, "use_for_cls_svp_dcp_ballanced"] = 1

    total = len(selected_idx)
    ratio = len(comb_idx) / total if total > 0 else 0
    print(f"{disease}: pacienti={df.loc[selected_idx, 'patient_id'].nunique()}/{n_dr}, combined={len(comb_idx)}, svp_only={len(svp_idx)}, total={total}, dcp_ratio={ratio:.2f}")

mask_bal = df["use_for_cls_svp_dcp_ballanced"] == 1

for disease in VALID_DISEASES:
    sub      = df[(df["label_clean_5class"] == disease) & mask_bal]
    patients = sub["patient_id"].unique()
    train_p, temp_p = train_test_split(patients.to_numpy(), test_size=0.30, random_state=42)
    test_p,  val_p  = train_test_split(temp_p,              test_size=0.50, random_state=42)
    df.loc[sub[sub["patient_id"].isin(train_p)].index, "split_svp_dcp_ballanced"] = "train"
    df.loc[sub[sub["patient_id"].isin(test_p)].index,  "split_svp_dcp_ballanced"] = "test"
    df.loc[sub[sub["patient_id"].isin(val_p)].index,   "split_svp_dcp_ballanced"] = "val"

print(df[mask_bal].groupby(["label_clean_5class", "split_svp_dcp_ballanced"]).size().unstack(fill_value=0).to_string())

DR referencia: 261 unikátnych pacientov
DR: pacienti=261/261, combined=144, svp_only=144, total=288, dcp_ratio=0.50
AMD: pacienti=43/261, combined=43, svp_only=0, total=43, dcp_ratio=1.00
Healthy: pacienti=261/261, combined=261, svp_only=1, total=262, dcp_ratio=1.00
RVO: pacienti=65/261, combined=49, svp_only=49, total=98, dcp_ratio=0.50
split_svp_dcp_ballanced  test  train  val
label_clean_5class                       
AMD                         6     30    7
DR                         44    198   46
Healthy                    39    183   40
RVO                        15     73   10


## 7. Split: MAE (predtréning)

- `use_for_mae`: 0 ak je snímka v `test` splite v akomkoľvek predchádzajúcom rozdelení, inak 1
- `split_mae`: test (use_for_mae=0) + train/val 90/10 na úrovni pacientov, seed=42

In [10]:
test_mask = (
    (df["split_full"] == "test") |
    (df["split_svp_dcp"] == "test") |
    (df["split_svp_dcp_ballanced"] == "test")
)

df["use_for_mae"] = (~test_mask).astype(int)
df["split_mae"]   = ""
df.loc[test_mask, "split_mae"] = "test"

remaining = df[df["use_for_mae"] == 1]
train_p, val_p = train_test_split(remaining["patient_id"].unique().to_numpy(), test_size=0.10, random_state=42)
df.loc[remaining[remaining["patient_id"].isin(train_p)].index, "split_mae"] = "train"
df.loc[remaining[remaining["patient_id"].isin(val_p)].index,   "split_mae"] = "val"

print(df["split_mae"].value_counts().to_string())

split_mae
train    3348
test      446
val       377


## 8. Uloženie výsledku

In [11]:
df.to_excel(OUTPUT, index=False)
print(f"Uložené: {OUTPUT}")
print(f"Celkový počet riadkov: {len(df)}")
print(f"Stĺpce: {list(df.columns)}")

Uložené: C:\Users\klara\Desktop\Fianl\03_Model_Training\data\master_excels\master_table_new.xlsx
Celkový počet riadkov: 4171
Stĺpce: ['sample_id', 'dataset', 'patient_id', 'svp_path', 'dcp_path', 'has_dcp', 'label_raw', 'label_clean_5class', 'use_for_cls_full', 'split_full', 'use_for_cls_svp_dcp', 'split_svp_dcp', 'use_for_cls_svp_dcp_ballanced', 'split_svp_dcp_ballanced', 'use_for_mae', 'split_mae']
